In [68]:
import pandas as pd
df = pd.read_csv('../dane/fra_cleaned.csv', sep=';', encoding='ISO-8859-1')

missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data_summary = pd.DataFrame({
    'Liczba braków': missing_values,
    'Procent braków (%)': missing_percentage.round(2)
})

columns_with_nan = missing_data_summary[missing_data_summary['Liczba braków'] > 0].sort_values(by='Liczba braków', ascending=False)

display(columns_with_nan)

,Liczba braków,Procent braków (%)
Perfumer2,22727,94.45
Year,2037,8.47
mainaccord5,981,4.08
mainaccord4,388,1.61
mainaccord3,114,0.47
mainaccord2,13,0.05


In [69]:
df['Rating Value'] = df['Rating Value'].astype(str).str.replace(',', '.').astype(float)
df['Perfumer2'] = df['Perfumer2'].fillna('')
df['Perfumers'] = df.apply(
    lambda row: f"{row['Perfumer1']}, {row['Perfumer2']}" if row['Perfumer2'] != '' else row['Perfumer1'], 
    axis=1
)
df = df.drop(columns=['Perfumer1', 'Perfumer2'])
columns_to_lower = [
    'Top', 'Middle', 'Base', 
    'mainaccord1', 'mainaccord2', 'mainaccord3', 'mainaccord4', 'mainaccord5'
]

for column in columns_to_lower:
    df[column] = df[column].astype(str).str.lower().str.strip()
    df[column] = df[column].replace(['nan', 'unknown', 'none', ''], pd.NA)

display(df[['Perfume', 'Rating Value', 'Perfumers', 'Year']].head())

,Perfume,Rating Value,Perfumers,Year
0,accento-overdose-pride-edition,1.42,unknown,2022.0
1,classique-pride-2024,1.86,unknown,2024.0
2,classique-pride-2023,1.91,"natalie gracia-cetto, quentin bisch",2023.0
3,pride-edition-man,1.92,unknown,2019.0
4,le-male-pride-collector,1.93,francis kurkdjian,2020.0


In [70]:
missing_accords = df[df['mainaccord2'].isna() | df['mainaccord3'].isna()]
columns_to_show = ['Perfume', 'Brand', 'mainaccord1', 'mainaccord2', 'mainaccord3']
display(missing_accords[columns_to_show].tail(10))

missing_accords_df = df[df['mainaccord2'].isna() | df['mainaccord3'].isna()].copy()
columns_to_export = ['Perfume', 'Brand', 'mainaccord1', 'mainaccord2', 'mainaccord3']

,Perfume,Brand,mainaccord1,mainaccord2,mainaccord3
21939,verveine-agrumes-2015,l-occitane-en-provence,citrus,aromatic,NaN
22376,cedro-di-diamante,perris-monte-carlo,citrus,warm spicy,NaN
22737,fleur-d-oranger-intense,fragonard,white floral,citrus,NaN
23077,khashab-al-oud-bois-de-oud,abdul-samad-al-qurashi,oud,NaN,NaN
23100,mercedes-benz-cologne,mercedes-benz,citrus,aromatic,NaN
23633,cristal-zest,massimo-dutti,citrus,aromatic,NaN
23671,jaguar-for-men-oud,jaguar,woody,vanilla,NaN
23786,ohood,amouage,rose,powdery,NaN
23831,dahn-oud-ateeque,abdul-samad-al-qurashi,oud,NaN,NaN
23924,oud-precieux,reine-de-saba,oud,NaN,NaN


In [71]:
import numpy as np

updated_accords_df = pd.read_excel('../dane/missing_accords_complete.xlsx', index_col=0)

cols_to_clean = ['mainaccord1', 'mainaccord2', 'mainaccord3']
for col in cols_to_clean:
    updated_accords_df[col] = updated_accords_df[col].astype(str).str.lower().str.strip()
    updated_accords_df[col] = updated_accords_df[col].replace(['nan', 'none', ''], np.nan)

for index, row in updated_accords_df.iterrows():
    for col in cols_to_clean:
        updated_accords_df.at[index, col] = row[col]

missing_accords2 = updated_accords_df['mainaccord2'].isna().sum()
missing_accords3 = updated_accords_df['mainaccord3'].isna().sum()

print(f"Brakujące wartości w mainaccord2 po aktualizacji: {missing_accords2}")
print(f"Brakujące wartości w mainaccord3 po aktualizacji: {missing_accords3}")

Brakujące wartości w mainaccord2 po aktualizacji: 0
Brakujące wartości w mainaccord3 po aktualizacji: 8


In [72]:
initial_count = len(df)
mask_pride = (
    df['Perfume'].str.contains('pride', regex=True, na=False) |
    df['Perfume'].str.contains('Pride', regex=True, na=False)
)

df = df[~mask_pride]
df = df.reset_index(drop=True)

final_count = len(df)
removed_count = initial_count - final_count

print(f"Usunięto {removed_count} edycji duplikatow z bazy danych.")
print(f"Obecna liczba perfum w bazie: {final_count}")

Usunięto 17 edycji duplikatow z bazy danych.
Obecna liczba perfum w bazie: 24046


In [73]:
df['Perfume'] = df['Perfume'].str.replace('-', ' ').str.upper()
df['Brand'] = df['Brand'].str.replace('-', ' ').str.upper()

In [74]:
df.head(25)

,url,Perfume,Brand,Country,Gender,Rating Value,Rating Count,Year,Top,Middle,Base,mainaccord1,mainaccord2,mainaccord3,mainaccord4,mainaccord5,Perfumers
0,https://www.fragrantica.com/perfume/we-pink/wa...,WAFFLE,WE PINK,Brazil,women,2.41,27,2023.0,"strawberry, almond, raspberry","butter, peach blossom, violet","praline, vanilla, musk",powdery,musky,violet,fruity,fresh,unknown
1,https://www.fragrantica.com/perfume/vivamor-pa...,ROUGE IMPERIALE,VIVAMOR PARFUMS,France,unisex,2.62,26,2023.0,"saffron, bulgarian rose","australian sandalwood, iris, jasmine","suede, bourbon vanilla, texas cedar",leather,warm spicy,rose,powdery,metallic,chris maurice
2,https://www.fragrantica.com/perfume/brocard/ch...,CHERRY LADY,BROCARD,Russia,women,2.64,39,NaN,"watermelon, mango, papaya, pineapple, peach, l...","lotus, hibiscus","musk, sandalwood, amber",fruity,tropical,sweet,ozonic,aquatic,unknown
3,https://www.fragrantica.com/perfume/bybozo/fre...,FRENCH KISS,BYBOZO,France,women,2.65,26,2023.0,"lemon verbena, neroli","fig, corn, magnolia, saffron, barberry, cherry...","pear wood, meringue, amber",fruity,woody,sweet,floral,citrus,unknown
4,https://www.fragrantica.com/perfume/sospiro-pe...,ERBA OUD,SOSPIRO PERFUMES,Italy,unisex,2.66,65,2022.0,"orange, bergamot, lemon","melon, saffron, peach, apple","agarwood (oud), leather, vanilla, amber, white...",citrus,oud,fruity,leather,sweet,christian provenzano
5,https://www.fragrantica.com/perfume/trump/dona...,DONALD TRUMP,TRUMP,USA,men,2.68,569,2004.0,"cucumber, black basil, mint, citruses","spicy notes, pepper, herbal notes, woody notes","spicy notes, exotic woods, vetiver",green,fresh spicy,aromatic,ozonic,aquatic,annie buzantian
6,https://www.fragrantica.com/perfume/rook-perfu...,SCHOOL,ROOK PERFUMES,UK,unisex,2.68,28,2021.0,"concrete, woody notes, ozonic notes, moss","rubber, paper, olibanum","coffee, cedar",rubber,mineral,paper,woody,NaN,nadeem crowe
7,https://www.fragrantica.com/perfume/o-boticari...,GLAMOUR GOLD GLAM,O BOTICARIO,Brazil,women,2.69,118,2022.0,"raspberry, neroli, bergamot","jasmine, orange blossom, tuberose","clearwood, amber, sandalwood, patchouli, cistu...",woody,white floral,fruity,sweet,amber,adilson rato
8,https://www.fragrantica.com/perfume/clean/clea...,CLEAN SIMPLY SOAP,CLEAN,USA,women,2.71,143,2009.0,"bergamot, frangipani","lotus, violet, rose",musk,floral,powdery,citrus,musky,violet,unknown
9,https://www.fragrantica.com/perfume/michael-ma...,VISIONAIR MIDNIGHT BLUE,MICHAEL MALUL LONDON,USA,men,2.71,42,2020.0,"watermelon, lavender, bergamot","lotus, jasmine","musk, sandalwood, amber",ozonic,aquatic,lavender,citrus,floral,unknown


In [75]:
mask_parentheses = (
    df['Top'].str.contains(r'\(|\)', regex=True, na=False) |
    df['Middle'].str.contains(r'\(|\)', regex=True, na=False) |
    df['Base'].str.contains(r'\(|\)', regex=True, na=False)
)

records_with_parentheses = df[mask_parentheses]
columns_to_inspect = ['Perfume', 'Brand', 'Top', 'Middle', 'Base']

print(f"Znaleziono {len(records_with_parentheses)} perfum z nawiasami w piramidzie zapachowej.")

Znaleziono 2024 perfum z nawiasami w piramidzie zapachowej.


In [76]:
notes_with_parentheses_set = set()
note_columns = ['Top', 'Middle', 'Base']

for col in note_columns:
    valid_rows = df[col].dropna()
    
    for text_block in valid_rows:
        split_notes = text_block.split(',')
        
        for note in split_notes:
            cleaned_note = note.strip()
            
            if '(' in cleaned_note or ')' in cleaned_note:
                notes_with_parentheses_set.add(cleaned_note)

unique_notes_with_parentheses = sorted(list(notes_with_parentheses_set))

print(f"Znaleziono {len(unique_notes_with_parentheses)} unikalnych nut z nawiasami.")
for note in unique_notes_with_parentheses:
    print(f"- {note}")

Znaleziono 15 unikalnych nut z nawiasami.
- agarwood (oud)
- ambrette (musk mallow)
- arbutus (madrona
- bearberry tree)
- carambola (star fruit)
- confetti (sugared almonds)
- cottonwood (poplar)
- erigeron (fleabane)
- geranium macrorrhizum (zdravetz)
- lime (linden blossom)
- lime (linden) blossom
- nard himalayan (jatamansi)
- poplar (populus) buds
- skeleton flower (diphylleia grayi)
- turnera diffusa (damiana)


In [77]:
replacements = {
    'agarwood (oud)': 'agarwood',
    'ambrette (musk mallow)': 'ambrette',
    'arbutus (madrona, bearberry tree)': 'bearberry tree',
    'carambola (star fruit)': 'carambola',
    'confetti (sugared almonds)': 'sugared almonds',
    'cottonwood (poplar)': 'cottonwood',
    'erigeron (fleabane)': 'erigeron',
    'geranium macrorrhizum (zdravetz)': 'geranium',
    'lime (linden blossom)': 'linden blossom',
    'lime (linden) blossom': 'linden blossom',
    'nard himalayan (jatamansi)': 'jatamansi',
    'poplar (populus) buds': 'poplar buds',
    'skeleton flower (diphylleia grayi)': 'skeleton flower',
    'turnera diffusa (damiana)': 'damiana'
}

columns_to_fix = ['Top', 'Middle', 'Base']

for col in columns_to_fix:
    for old_val, new_val in replacements.items():
        df[col] = df[col].str.replace(old_val, new_val, regex=False)

In [78]:
import re
import unicodedata
from collections import Counter
from IPython.display import display

In [79]:
note_columns = ['Top', 'Middle', 'Base']

accord_columns = [
    'mainaccord1',
    'mainaccord2',
    'mainaccord3',
    'mainaccord4',
    'mainaccord5'
]

columns_to_check = note_columns + accord_columns

value_counts = Counter()

for col in columns_to_check:
    for cell in df[col].dropna():
        for item in str(cell).split(','):
            item = item.strip()

            if item:
                value_counts[item] += 1

unique_values = pd.DataFrame(
    value_counts.items(),
    columns=['name', 'count']
)

unique_values = unique_values.sort_values(
    'count',
    ascending=False
).reset_index(drop=True)

print("Liczba unikalnych nazw:", len(unique_values))

display(unique_values.head(30))

Liczba unikalnych nazw: 1708


,name,count
0,amber,12869
1,woody,12523
2,vanilla,11379
3,musk,11013
4,citrus,9492
5,rose,9478
6,bergamot,8613
7,patchouli,8553
8,sandalwood,8057
9,jasmine,7741


In [80]:
allowed_punctuation = set(" -'’./&+")

def detect_anomalies(value):
    flags = []

    if re.search(r'[™®©℠]', value):
        flags.append('symbol_towarowy')

    if any(
        unicodedata.category(ch) in {'Cc', 'Cf', 'Cs'}
        for ch in value
    ):
        flags.append('znak_niewidoczny_lub_sterujacy')

    unusual = sorted({
        ch for ch in value
        if not (
            ch.isalnum()
            or ch.isspace()
            or ch in allowed_punctuation
        )
    })

    if unusual:
        flags.append(
            'nietypowe_znaki:' + ''.join(unusual)
        )

    if value != value.strip():
        flags.append('spacje_na_brzegach')

    if re.search(r'\s{2,}', value):
        flags.append('wielokrotne_spacje')

    return flags


audit = unique_values.copy()

audit['anomalies'] = audit['name'].apply(
    detect_anomalies
)

suspicious = audit[
    audit['anomalies'].apply(bool)
].copy()

print("Liczba podejrzanych nazw:", len(suspicious))

display(
    suspicious.sort_values(
        'count',
        ascending=False
    ).reset_index(drop=True)
)

Liczba podejrzanych nazw: 20


,name,count,anomalies
0,accord eudora®,21,"[symbol_towarowy, nietypowe_znaki:®]"
1,orcanox,15,"[znak_niewidoczny_lub_sterujacy, nietypowe_zna..."
2,ambrofix,10,"[znak_niewidoczny_lub_sterujacy, nietypowe_zna..."
3,dyers greenweed,10,"[znak_niewidoczny_lub_sterujacy, nietypowe_zna..."
4,pepperwood,5,"[znak_niewidoczny_lub_sterujacy, nietypowe_zna..."
5,scent trek®,4,"[symbol_towarowy, nietypowe_znaki:®]"
6,hivernal®,3,"[symbol_towarowy, nietypowe_znaki:®]"
7,norlimbanol,3,"[znak_niewidoczny_lub_sterujacy, nietypowe_zna..."
8,nympheal,2,"[znak_niewidoczny_lub_sterujacy, nietypowe_zna..."
9,lilybelle®,2,"[symbol_towarowy, nietypowe_znaki:®]"


In [81]:

TRADEMARKS = str.maketrans('', '', '™®©℠')

def clean_name(value):
    if not isinstance(value, str):
        return value

    value = unicodedata.normalize('NFKC', value)

    # Usunięcie znaków towarowych
    value = value.translate(TRADEMARKS)

    value = value.replace('\x92', "'")

    value = value.replace('’', "'").replace('‘', "'")

    value = ''.join(
        ch for ch in value
        if unicodedata.category(ch) != 'Cf'
    )

    value = ''.join(
        ch for ch in value
        if unicodedata.category(ch) != 'Cc'
        or ch in '\t\n\r'
    )

    value = re.sub(r'\s+', ' ', value).strip()

    return value


In [82]:
df_clean = df.copy()

def clean_list_cell(value):
    if not isinstance(value, str):
        return value

    items = [
        clean_name(item)
        for item in value.split(',')
    ]

    result = []
    seen = set()

    for item in items:
        if not item:
            continue

        key = item.casefold()

        if key not in seen:
            seen.add(key)
            result.append(item)

    return ', '.join(result)


for col in columns_to_check:
    df_clean[col] = df_clean[col].apply(clean_list_cell)

print("Czyszczenie zakończone.")
display(df_clean[columns_to_check].head(10))

Czyszczenie zakończone.


,Top,Middle,Base,mainaccord1,mainaccord2,mainaccord3,mainaccord4,mainaccord5
0,"strawberry, almond, raspberry","butter, peach blossom, violet","praline, vanilla, musk",powdery,musky,violet,fruity,fresh
1,"saffron, bulgarian rose","australian sandalwood, iris, jasmine","suede, bourbon vanilla, texas cedar",leather,warm spicy,rose,powdery,metallic
2,"watermelon, mango, papaya, pineapple, peach, l...","lotus, hibiscus","musk, sandalwood, amber",fruity,tropical,sweet,ozonic,aquatic
3,"lemon verbena, neroli","fig, corn, magnolia, saffron, barberry, cherry...","pear wood, meringue, amber",fruity,woody,sweet,floral,citrus
4,"orange, bergamot, lemon","melon, saffron, peach, apple","agarwood, leather, vanilla, amber, white musk",citrus,oud,fruity,leather,sweet
5,"cucumber, black basil, mint, citruses","spicy notes, pepper, herbal notes, woody notes","spicy notes, exotic woods, vetiver",green,fresh spicy,aromatic,ozonic,aquatic
6,"concrete, woody notes, ozonic notes, moss","rubber, paper, olibanum","coffee, cedar",rubber,mineral,paper,woody,NaN
7,"raspberry, neroli, bergamot","jasmine, orange blossom, tuberose","clearwood, amber, sandalwood, patchouli, cistu...",woody,white floral,fruity,sweet,amber
8,"bergamot, frangipani","lotus, violet, rose",musk,floral,powdery,citrus,musky,violet
9,"watermelon, lavender, bergamot","lotus, jasmine","musk, sandalwood, amber",ozonic,aquatic,lavender,citrus,floral


In [83]:
col = 'Middle'

changed = df[col].astype(str) != df_clean[col].astype(str)

display(
    pd.DataFrame({
        'before': df.loc[changed, col],
        'after': df_clean.loc[changed, col]
    }).head()
)

,before,after
3390,"green tea, petalia, karmaflor®, mahonial","green tea, petalia, karmaflor, mahonial"
4374,"dyers greenweed, lily, floral notes","dyer's greenweed, lily, floral notes"
6309,"orange blossom, petalia, lily-of-the-valley, v...","orange blossom, petalia, lily-of-the-valley, v..."
9507,"rose, syringa, dyers greenweed","rose, syringa, dyer's greenweed"
9830,"freesia, cashmere wood, fig blossom, cedar, ny...","freesia, cashmere wood, fig blossom, cedar, ny..."
